# Technical note: Checking for cycles in the system of equations

This note discusses cycle checking in Myokit's representation of a system of equations.

Three levels are discussed:

1. An expression that references itself.
2. A variable that references itself directly, in its defining expression or initial value.
3. A variable that references itself indirectly, through its defining expression or initial value, and at least one other variable.

The notes include an audit of where validation (and cache invalidation) is performed, based on `Myokit 1.40.0`.

## Case 1: A self-referencing expression

Expressions should be structured as [trees](https://en.wikipedia.org/wiki/Tree_(graph_theory)).
This means that there should be no cycles.
More strongly, there should be exactly one path between any two nodes in the tree.

To guarantee this, Myokit expressions are defined as **immutable**, with their operands set at construction time.

As a result, direct self-reference, e.g. `x = 1 + x`, is not possible through the API.
```
x = Plus(x, y)  # redefines x, no self-reference!
```

But can be achieved by mutating an expression _outside of the API_:
```
x = Plus(Number(1), Number(1))
x._operands[1] = x
```

## Case 2: Direct self-reference



# Questions

- How do we avoid unnecessary validation
  - Is it cached? Yes for expr. No for model status
  - When is it called?
- When are initial value expressions evaluated?
- Can we make cycles with `eval(subst=...)`
- Where do we check that initial values only reference (1) non-nested; and (2) constants.

## Implementation

The checks for cycles are run by

- Case 1: `Expression.validate()` --> But to be changed
- Case 2: `Model._validate_solvability()`
- Case 3: `Model._validate_solvability()`

lalala

- In `.validate()` methods, which are defined on `Expression`, `Variable`, `Component`, and `Model`
  - `Model.validate` calls `Component.validate`, which calls `Variable.validate`, which calls `.validate` on any child variables.
  - `Variable.validate` calls `Variable._rhs.validate` but does **not** call `init.validate()`

lololo

- In **XXXXXX**






## When is model.validate() called

1. When simulations are created
2. When HH or Markov models are created
3. Before exporting
4. After parsing (sets unused variable warnings)
5. Before creating python functions from Myokit equations: `Variable.pyfunc()`
6. When evaluating an RHS: `Model.show_evaluation_of`, `Variable.clamp`,
   - But **not** by `Variable.eval()`!
   - And **not** by `Expression.eval()`!
7. When evaluating derivatives: `Model.evaluate_derivatives` calls `solvable_order` 
8. When evaluating initial values: `Model.initial_values(as_floats=True)`, `Variable.initial_value(True)`

The last 2 can occur quite easily, e.g. when calling `float(var)`

## Validation required from within the expression system?

# TODO CHECK

- `eval()`
- `eval(subst)`
- `eval_unit()`
- `diff()`
- `diff()`-
- `depends_on()`
- `depends_on_state()`

# TODO CHECK FOR MORE

## Caching and invalidation

Full model validation (including solvability) is cached in `Model._is_valid` which is used as a tribool (`True`, `False`, `None`).

However, finding a model to be invalid _also_ raises a `myokit.IntegrityError`, so that the `False` value is rarely used.

The model validation status is reset by

- `Model.add_component` (and `add_component_allow_renaming`), `Model.remove_component`
- `Model.set_initial_values(x)` if `x` is a string or dict (handled by `map_to_state`)
- `Model._reset_validation()`, called by
  - `VarOwner.add_variable`, `add_variable_allow_renaming`
  - `VarOwner.move_variable`, `VarOwner.remove_variable`
  - `Variable.promote`, `Variable.demote`
  - `Variable.remove_child_variables()`
  - `Variable.set_binding` (bound variables are independent)
  - `Variable.set_rhs` and `Variable._set_initial_value`
  

Indirectly, it is set by methods such as `Variable.clamp` and others that call the above methods.



# TODO

- [x] ~Change expression validation of privates to asserts https://github.com/myokit/myokit/issues/1235~
- [x] `Variable.validate()` should validate initial value as well as RHS
- [ ] is_derivative missing () https://github.com/myokit/myokit/issues/1239
- [ ] Remove expression.validate() https://github.com/myokit/myokit/issues/1234
- [ ] Validate before eval() https://github.com/myokit/myokit/issues/1238
- [ ] Variable reffed only by init is marked unused https://github.com/myokit/myokit/issues/1163
- [ ] Bug in remove_child_variables() https://github.com/myokit/myokit/issues/1241
- [ ] Bug in Variable._delete https://github.com/myokit/myokit/issues/1242
- [ ] Reset validation after remove_child_variables
- [ ] Cache model validation
- [ ] Benchmark validation before and after changes

# WHERE IS INIT USED

## When exporting with "true"

```
./myokit/formats/ansic/template/euler.c:    print(tab + v(var) + ' = ' + myokit.float.str(var.initial_value(True)) + ';')

./myokit/formats/easyml/_exporter.py:                        + myokit.float.str(v.initial_value(True)) + eos)

./myokit/formats/opencl/template/cable.c:    print(tab*2 + '*s = ' + myokit.float.str(v.initial_value(True)) + '; s++; // ' + str(v.qname()))
```

**To-do**: Is it OK that the other exporters don't do this?

## Exporting without "true"

```
./myokit/formats/diffsl/_exporter.py:            rhs = e.ex(v.initial_value())
```

## Converting to HH/Markov

```
./myokit/lib/hh.py:            [v.initial_value(True) for v in self._states])
./myokit/lib/markov.py:            [v.initial_value(True) for v in self._states])
```

## Creating a CModel

```
./myokit/_sim/cmodel.h:    print(tab + v(var) + ' = ' + myokit.float.str(var.initial_value(True)) + ';')
```

## OTHER OTHER OTHER

```
./myokit/_model_api.py:                        [e.var() for e in var.initial_value().references()])
./myokit/_model_api.py:                state_map[var].initial_value().clone(subst=lhs_map))
./myokit/_model_api.py:            value = var.initial_value()
./myokit/_model_api.py:                value = self.initial_value(True)
./myokit/_model_api.py:            value = self.initial_value()
./myokit/_model_api.py:        return self.initial_value(as_float=True)
```

### Case 1: Check for cycles within a single expression

At the time of writing (Myokit 1.40.0) the `Expression.validate()` method exists, mainly to check for "case 1" cycles.

1. The unit test for this changes the private properties of an expression, as in the example above
2. The result of the check is cached, so validation only happens once (which relies on people not messing, so unsafe!)

This should probably go: https://github.com/myokit/myokit/issues/1234

## Warnings

These are only raised for unused variables, in `Model._validate_solvability`.